---
## 1. Library Imports

This section imports all required libraries:

- **`chardet`** — automatic CSV file encoding detection (important for files with Cyrillic characters)
- **`pandas` / `numpy`** — core tools for tabular data processing and numerical computation
- **`matplotlib` / `seaborn`** — result visualization
- **`scipy.sparse`** — sparse matrix operations (TF-IDF produces matrices with millions of zero elements)
- **`sklearn`** — full ML stack: text vectorization, feature scaling, classifiers, metrics

Constants `RANDOM_STATE` and `N_QUINTILES` fix reproducibility and the number of target classes.

In [ ]:
import chardet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix
from sklearn.decomposition import TruncatedSVD

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, label_binarize, OneHotEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

RANDOM_STATE = 42
N_QUINTILES  = 5
print('All libraries imported successfully')

---
## 2. Data Loading

The file is read in two steps:

1. **Encoding detection** (`chardet.detect`): we read the first 100 KB of the file in binary mode and automatically detect the encoding (e.g. `windows-1251` or `utf-8`). This is critical for Ukrainian-language texts where wrong encoding produces unreadable characters.

2. **CSV reading** with `;` delimiter and automatic date parsing in the `receivedDateTime` column. Pandas converts date strings to `datetime64` objects, making it easy to extract hour, day of week, etc.

After loading, we print the dataset shape and first rows for an initial overview.

In [ ]:
with open('./data/appeals_2026-04-02.csv', 'rb') as f:
    enc = chardet.detect(f.read(100000))
print(enc)
        
df_raw = pd.read_csv(
    './data/appeals_2026-04-02.csv',
    encoding=enc['encoding'],
    sep=';',
    parse_dates=['receivedDateTime']
)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
df_raw["result"].unique()

In [ ]:
df_raw.describe()

In [ ]:
df_raw.dtypes

---
## 3. Target Variable Construction — Review Result

**Target variable**: `result` — the actual outcome of an appeal review.

### NaN Handling
`NaN` in the `result` field means the appeal **has not been reviewed yet**.  
Such records are **excluded** from the dataset — we only predict for already-reviewed appeals.

### Label Encoding
Unique `result` values are encoded into numeric labels via `LabelEncoder`.  
The class distribution is shown in the histogram below.

In [ ]:
# Filter: keep only reviewed appeals (result is not NaN)
df_resolved = df_raw.dropna(subset=['result']).copy()
print(f'Reviewed: {len(df_resolved):,} / {len(df_raw):,}')
print(f'Unreviewed (NaN result) excluded: {len(df_raw) - len(df_resolved):,}')

# Unique result values
print('\nUnique result values:')
print(df_resolved['result'].value_counts())

fig, ax = plt.subplots(figsize=(10, 4))
counts = df_resolved['result'].value_counts()
bars = ax.bar(range(len(counts)), counts.values,
              color=['#27ae60','#2980b9','#f39c12','#e67e22','#e74c3c'][:len(counts)],
              alpha=0.88, edgecolor='white')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 5, str(v), ha='center', fontsize=9)
ax.set_title('Target variable: Distribution of review results')
ax.set_xlabel('Result')
ax.set_ylabel('Number of appeals')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=20, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

Features are split into **two fully isolated sets**:

| Strategy | Features | Used in |
|---|---|---|
| **Text only** | TF-IDF on `type + kind + content` | NB, LR |
| **Tabular only** | Temporal + address + organizational — *zero TF-IDF* | RF, HistGB |

### Temporal Features
`hour`, `dayofweek`, `month`, `is_weekend`, `is_night`, `is_business_hours`,  
sine-cosine encoding of hour and day of week.  
**Note**: `weekofyear` is **excluded** — it is monotonically increasing and correlates with the time split.

### Address and Organizational Features
`postcode_known`, `street_len`, `has_building`, `org_id_str`, `content_len`.

### FIX 4: Frequency Encoding for `org_id_str`
Instead of OHE (which generates thousands of columns and memorizes specific ids), we use  
**Frequency Encoding** — each organization is replaced by its appeal count in train.  
This preserves information about 'large' and 'small' organizations without overfitting to ids.

In [ ]:
df = df_resolved.copy()

# Temporal features (weekofyear EXCLUDED — monotonic proxy of the time split)
df['hour']              = df['receivedDateTime'].dt.hour
df['dayofweek']         = df['receivedDateTime'].dt.dayofweek
df['month']             = df['receivedDateTime'].dt.month
df['is_weekend']        = (df['dayofweek'] >= 5).astype(int)
df['is_night']          = ((df['hour'] < 7) | (df['hour'] >= 22)).astype(int)
df['is_business_hours'] = ((df['hour'] >= 9) & (df['hour'] <= 17)
                            & (df['dayofweek'] < 5)).astype(int)
df['hour_sin']          = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']          = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']           = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']           = np.cos(2 * np.pi * df['dayofweek'] / 7)

# Address features
df['postcode_known']  = df['addressPostCode'].notna().astype(int)
df['postcode']        = df['addressPostCode'].fillna(0).astype(int).astype(str)
df['street_len']      = df['addressThoroughfare'].fillna('').str.len()
df['has_building']    = df['addressLocatorDesignator'].notna().astype(int)

# Organization and content features
df['org_id_str']   = df['organizationId'].fillna(0).astype(int).astype(str)
df['content_len']  = df['content'].fillna('').str.len()

# Text for TF-IDF models
df['input_text'] = (df['type'].fillna('') + ' | ' +
                    df['kind'].fillna('') + ' | ' +
                    df['content'].fillna(''))

# Target variable: result (string labels -> numeric)
TARGET = 'result'
le = LabelEncoder()
df['label'] = le.fit_transform(df[TARGET])
classes = le.classes_
class_names = list(classes)

print(f'Classes ({len(classes)}): {list(classes)}')
print(f'\nLabel distribution:')
print(df['label'].value_counts().rename(index=dict(enumerate(classes))))
print(df[['hour','dayofweek','is_weekend','is_night','postcode_known',
          'street_len','content_len']].describe().round(2))

---
## 5. Temporal Train/Test Split

**Temporal split**: train on January–February 2026, test on March 2026.

**Important**: organizational aggregates are computed only on the training set  
(to avoid leakage via `result`). We use **structural aggregates**:  
organization appeal count (`org_volume`) and per-category appeal frequency (`org_kind_count`).

**Frequency Encoding** for `org_id_str` is also computed here — exclusively on train.

In [ ]:
train_df = df[df['month'] <= 2].copy()
test_df  = df[df['month'] == 3].copy()

print(f'Training set (January-February): {len(train_df):,} records')
print(f'Test set      (March)           : {len(test_df):,} records')

assert train_df['receivedDateTime'].max() < test_df['receivedDateTime'].min(), \
    'Temporal leakage detected!'
print('No temporal leakage detected ')

# Aggregates computed only on training data (no target variable used — no leakage)
org_volume = (
    train_df
    .groupby('org_id_str')
    .size()
    .reset_index(name='org_volume')
)

org_kind_counts = (
    train_df
    .groupby(['org_id_str', 'kind'])
    .size()
    .reset_index(name='org_kind_count')
)

#  FIX 4: Frequency Encoding for org_id_str (train only)
# Instead of OHE, which creates thousands of columns and memorizes specific ids,
# we replace each organization with its appeal count in train.
# New organizations in test will get value 0 (instead of an all-zero OHE vector).
org_freq_map = train_df['org_id_str'].value_counts().to_dict()

def add_agg_features(dframe):
    d = dframe.copy()
    d = d.merge(org_volume, on='org_id_str', how='left')
    d = d.merge(org_kind_counts, on=['org_id_str', 'kind'], how='left')
    d['org_volume'].fillna(0, inplace=True)
    d['org_kind_count'].fillna(0, inplace=True)
    # Frequency encoding: org_freq = appeal count of this org in train
    d['org_freq'] = d['org_id_str'].map(org_freq_map).fillna(0)
    return d

train_df = add_agg_features(train_df)
test_df  = add_agg_features(test_df)

print(f'\nOrganization volume (top by appeal count):')
print(org_volume.sort_values('org_volume', ascending=False).head(6))
print(f'\nFrequency Encoding — new orgs in test (will get 0): '
      f"{(test_df['org_freq'] == 0).sum()}")

---
## 6. Feature Matrix Construction

Five matrices for different models:

1. **TF-IDF** — character n-gram (2–4) on concatenated text, `min_df=5` (fix #5)
2. **Sparse tabular** — numeric (scaled) + OHE only for `type`, `kind`, `postcode`  
   (without `org_id_str` — it is now in `org_freq` as a numeric feature)
3. **Dense tabular** — for HistGradientBoosting
4. **Sparse fused** — TF-IDF + tabular (for LogReg)
5. **Dense fused (SVD)** — SVD(100) of text + tabular **← new model!**

### FIX 3: `X_train_fused_dense` is now used in a model
### FIX 5: `min_df=5` instead of `min_df=1`

In [ ]:
#  FIX 5: min_df=5 — remove tokens occurring fewer than 5 times
# (in the old version min_df=1 introduced noise from unique names and typos)
tfidf = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 4),
    max_features=15000, sublinear_tf=True,
    min_df=5  # ← fixed from 1 (default)
)
X_train_tfidf = tfidf.fit_transform(train_df['input_text'])
X_test_tfidf  = tfidf.transform(test_df['input_text'])

y_train = train_df['label'].values
y_test  = test_df['label'].values

print(f'TF-IDF: train {X_train_tfidf.shape}, test {X_test_tfidf.shape}')

#  FIX 4: org_id_str removed from CAT_COLS — now in org_freq as a numeric feature
NUM_COLS = [
    'hour', 'dayofweek', 'is_weekend', 'is_night', 'is_business_hours',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'postcode_known', 'street_len', 'has_building', 'content_len',
    'org_volume', 'org_kind_count',
    'org_freq'  # ← new feature: Frequency Encoding instead of OHE org_id_str
]
# org_id_str removed from CAT_COLS
CAT_COLS = ['type', 'kind', 'postcode']

for col in CAT_COLS:
    train_df[col] = train_df[col].fillna('UNKNOWN')
    test_df[col]  = test_df[col].fillna('UNKNOWN')

# Categorical encoding (only type, kind, postcode)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = ohe.fit_transform(train_df[CAT_COLS])
X_test_cat  = ohe.transform(test_df[CAT_COLS])

# Numeric feature scaling
scaler = StandardScaler(with_mean=False)
X_train_num = csr_matrix(scaler.fit_transform(
    train_df[NUM_COLS].fillna(0).values))
X_test_num  = csr_matrix(scaler.transform(
    test_df[NUM_COLS].fillna(0).values))

# 3. Tabular matrix (sparse and dense)
X_train_tab = hstack([X_train_num, X_train_cat])
X_test_tab  = hstack([X_test_num,  X_test_cat])

X_train_tab_dense = np.hstack([
    train_df[NUM_COLS].fillna(np.nan).values,
    X_train_cat.toarray()
])
X_test_tab_dense = np.hstack([
    test_df[NUM_COLS].fillna(np.nan).values,
    X_test_cat.toarray()
])

# 4. Fused matrix (TF-IDF + tabular — sparse, for LogReg)
#  FIX 6: TF-IDF scaling in the fused matrix
# Problem: TF-IDF has 3616 columns vs 75 tabular → takes 98% of L2 norm,
# forcing LogReg to ignore tabular features (96.4% of weights go to text).
# Solution: multiply TF-IDF by alpha = sqrt(n_tab / n_tfidf),
# so both blocks have equal 'voice' in the regularized model.
import scipy.sparse as sp
_n_tfidf = X_train_tfidf.shape[1]
_n_tab   = X_train_tab.shape[1]
_tfidf_scale = float(np.sqrt(_n_tab / _n_tfidf))  # ≈ 0.144
print(f'TF-IDF scale factor: {_tfidf_scale:.4f}  '
      f'(n_tfidf={_n_tfidf}, n_tab={_n_tab})')
X_train_tfidf_scaled = X_train_tfidf.multiply(_tfidf_scale)
X_test_tfidf_scaled  = X_test_tfidf.multiply(_tfidf_scale)
X_train_fused = hstack([X_train_tfidf_scaled, X_train_tab])
X_test_fused  = hstack([X_test_tfidf_scaled, X_test_tab])

# 5.  FIX 3: Dense fused matrix (text SVD + tabular)
# In the previous version X_train_fused_dense was defined but never used!
# Now it is passed to the new HistGB model (Dense fused SVD).
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_train_text_dense = svd.fit_transform(X_train_tfidf)
X_test_text_dense  = svd.transform(X_test_tfidf)

X_train_fused_dense = np.hstack([X_train_tab_dense, X_train_text_dense])
X_test_fused_dense  = np.hstack([X_test_tab_dense, X_test_text_dense])

print(f'Tabular (sparse): train {X_train_tab.shape}')
print(f'Tabular (dense) : train {X_train_tab_dense.shape}')
print(f'Fused  (sparse) : train {X_train_fused.shape}')
print(f'Fused  (dense SVD): train {X_train_fused_dense.shape}')
print(f'\nSVD: explains {svd.explained_variance_ratio_.sum()*100:.1f}% of TF-IDF variance')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Hour of day vs result distribution
ax = axes[0]
hour_result = train_df.groupby(['hour', 'result']).size().unstack(fill_value=0)
hour_result_pct = hour_result.div(hour_result.sum(axis=1), axis=0)
hour_result_pct.plot(kind='bar', stacked=True, ax=ax, legend=False, colormap='tab10')
ax.set_title('Result distribution by hour of day')
ax.set_xlabel('Hour')
ax.set_ylabel('Share')
ax.tick_params(axis='x', rotation=45)

# 2. Day of week vs results
ax = axes[1]
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_result = train_df.groupby(['dayofweek', 'result']).size().unstack(fill_value=0)
dow_result_pct = dow_result.div(dow_result.sum(axis=1), axis=0)
dow_result_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_xticks(range(7))
ax.set_xticklabels(day_names, rotation=0)
ax.set_title('Result distribution by day of week')
ax.set_ylabel('Share')
ax.legend(title='result', fontsize=7, loc='lower right')

plt.suptitle('Feature signal analysis — temporal patterns', fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Model Definitions

Six classifiers with clearly separated roles for predicting `result`:

### Text models
- **Naive Bayes**: baseline, binarized TF-IDF. `alpha=1.0` (fixed from 0.001)
- **Logistic Regression**: linear boundary in TF-IDF space, `class_weight='balanced'`

### Tabular models
- **Random Forest**: tree ensemble, `class_weight='balanced'`
- **HistGradientBoosting**: `max_iter=200, max_depth=4` (fixed from `10, 1`)

### Mixed models
- **LR (Sparse fused)**: TF-IDF + tabular matrix
- **HistGB (Dense fused SVD)**: SVD(100) + tabular — **new model**, fixes the unused-matrix bug

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import recall_score, make_scorer
import numpy as np
import math

def worst_class_recall(y_true, y_pred):
    return np.min(recall_score(y_true, y_pred, average=None, zero_division=0))

scorer = make_scorer(worst_class_recall)

#  FIX 7: sample_weight for HistGB (does not support class_weight)
# Class 0 (Resolved positively): 72.1%, class 1 (Explanation provided): 27.9%
# Without weighting HistGB always predicts majority → poor confusion matrix
from sklearn.utils.class_weight import compute_sample_weight
sample_weight_train = compute_sample_weight('balanced', y_train)

# ── Single model registry ────────────────────────────────────────────────
registry = [
    {
        'name': 'Naive Bayes (text only)',
        'model': MultinomialNB(),
        'grid': {'alpha': [0.1, 0.5, 1.0, 5.0]},
        'X_train': X_train_tfidf.multiply(X_train_tfidf > 0),
        'X_test':  X_test_tfidf.multiply(X_test_tfidf > 0),
    },
    {
        'name': 'Logistic Regression (text only)',
        'model': LogisticRegression(max_iter=1000, class_weight='balanced',
                                    solver='saga', penalty='elasticnet',
                                    random_state=RANDOM_STATE),
        'grid': {'C': [0.1, 1.0, 10.0, 100.0], 'l1_ratio': [0.15, 0.5, 0.8]},
        'X_train': X_train_tfidf,
        'X_test':  X_test_tfidf,
    },
    {
        'name': 'Random Forest (tabular only)',
        'model': RandomForestClassifier(class_weight='balanced',
                                        random_state=RANDOM_STATE, n_jobs=-1),
        'grid': {'n_estimators': [100, 200], 'max_depth': [6, 12, None],
                 'min_samples_leaf': [10, 20]},
        'X_train': X_train_tab,
        'X_test':  X_test_tab,
    },
    {
        'name': 'HistGB (tabular only)',
        'model': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        'grid': {'max_iter': [100, 200], 'max_depth': [4, 10],
                 'learning_rate': [0.01, 0.05, 0.1],
                 'l2_regularization': [0.1, 1.0, 10.0]},
        'X_train': X_train_tab_dense,
        'X_test':  X_test_tab_dense,
        'sample_weight': sample_weight_train,  # Fix 7: balanced weighting
    },
    {
        'name': 'Logistic Regression (Fused: Text + Tabular)',
        'model': LogisticRegression(max_iter=1000, class_weight='balanced',
                                    solver='saga', penalty='elasticnet',
                                    random_state=RANDOM_STATE),
        'grid': {'C': [0.1, 1.0, 10.0, 100.0], 'l1_ratio': [0.15, 0.5, 0.8]},
        'X_train': X_train_fused,
        'X_test':  X_test_fused,
    },
    {
        'name': 'HistGB (Dense fused: SVD + Tabular)',
        'model': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        'grid': {'max_iter': [100, 200], 'max_depth': [4, 10],
                 'learning_rate': [0.01, 0.05, 0.1],
                 'l2_regularization': [0.1, 1.0, 10.0]},
        'X_train': X_train_fused_dense,
        'X_test':  X_test_fused_dense,
        'sample_weight': sample_weight_train,  # Fix 7: balanced weighting
    },
]

def _grid_size(grid):
    return math.prod(len(v) for v in grid.values())

# ── Single tuning loop ──────────────────────────────────────────────────
print("Starting RandomizedSearch optimization...\n")
models = {}   # built here, replaces both old dicts

for cfg in registry:
    name = cfg['name']
    print(f" {name} ...", end=' ', flush=True)

    search = RandomizedSearchCV(
        estimator=cfg['model'],
        param_distributions=cfg['grid'],
        n_iter=min(20, _grid_size(cfg['grid'])),  # see helper function below
        cv=5,
        scoring={'worst_recall': scorer},
        refit='worst_recall',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
    )
    _sw = cfg.get('sample_weight', None)
    _fit_kw = {'sample_weight': _sw} if _sw is not None else {}
    search.fit(cfg['X_train'], y_train, **_fit_kw)

    models[name] = {
        'model':   search.best_estimator_,
        'X_train': cfg['X_train'],
        'X_test':  cfg['X_test'],
        'note':    f"Best params: {search.best_params_}",
    }
    print(f"score={search.best_score_:.4f}  {search.best_params_}")

---
## 8. Training and Evaluation

Each model is trained and evaluated on seven metrics.  
**Key metric** — Macro F1, which penalizes equally for poor results on any class.

**MAE baseline**: MAE is not applicable for predicting `result` (no ordering between classes).  
We use **Cohen's κ** as the primary balance metric instead.

In [ ]:
results = {}
predictions = {}
probas = {}

y_test_bin = label_binarize(y_test, classes=np.arange(len(classes)))

def worst_class_recall_eval(y_true, y_pred):
    recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
    return np.min(recalls)

for name, cfg in models.items():
    short = name.replace('\n', ' ')
    print(f'Training {short}...', end=' ', flush=True)

    model = cfg['model']
    Xtr   = cfg['X_train']
    Xte   = cfg['X_test']

    sw = cfg.get('sample_weight', None)
    if sw is not None:
        model.fit(Xtr, y_train, sample_weight=sw)
    else:
        model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    predictions[name] = y_pred

    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(Xte)
    else:
        y_proba = np.zeros((len(y_test), len(classes)))
        y_proba[np.arange(len(y_pred)), y_pred] = 1.0

    if hasattr(model, 'classes_'):
        y_proba_aligned = np.zeros((y_proba.shape[0], len(classes)))
        for i, cls_idx in enumerate(model.classes_):
            y_proba_aligned[:, cls_idx] = y_proba[:, i]
    else:
        y_proba_aligned = y_proba

    probas[name] = y_proba_aligned

    try:
        roc_auc = roc_auc_score(
            y_test, y_proba_aligned,
            multi_class='ovr', average='macro'
        )
    except Exception:
        roc_auc = np.nan

    results[name] = {
        'Accuracy':        accuracy_score(y_test, y_pred),
        'Macro F1':        f1_score(y_test, y_pred, average='macro'),
        'Weighted F1':     f1_score(y_test, y_pred, average='weighted'),
        'Macro Precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Macro Recall':    recall_score(y_test, y_pred, average='macro', zero_division=0),
        'Worst Recall':    worst_class_recall_eval(y_test, y_pred),  # <-- added
        'MCC':             matthews_corrcoef(y_test, y_pred),
        "Cohen's κ":       cohen_kappa_score(y_test, y_pred),
    }
    print(f"done. Acc={results[name]['Accuracy']:.3f}  "
          f"MacroF1={results[name]['Macro F1']:.3f}")

In [ ]:
results_df = pd.DataFrame(results).T.round(4)
print('\n=== Results on test set (March 2026) ===')

higher_is_better = list(results_df.columns)
cols = ["Accuracy", "Macro F1", "Worst Recall"]  # only these columns

display(
    results_df.style
        .highlight_max(subset=cols, axis=0, color="green")
        .highlight_min(subset=cols, axis=0, color="red")
        .format("{:.4f}")
)

---
## 9. Visual Model Comparison

Three charts provide different perspectives on the results:

1. **Metric bar charts**: direct comparison of Accuracy, Macro F1, MCC across models
2. **Cohen's κ**: key balance metric with baseline references
3. **Radar chart**: model profiles across all metrics simultaneously

In [ ]:
metrics_to_plot = ['Accuracy', 'Worst Recall']
model_labels = [n.replace('\n', '\n') for n in results_df.index]
palette = ['#e74c3c', '#e67e22', '#2980b9', '#27ae60', '#8e44ad', '#16a085']

# Changed from (1, N) → (N, 1)
fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(10, 10))

# Ensure axes is iterable when there is only one subplot
if len(metrics_to_plot) == 1:
    axes = [axes]

for ax, metric in zip(axes, metrics_to_plot):
    vals = results_df[metric].values.astype(float)
    bars = ax.bar(range(len(vals)), vals,
                  color=palette[:len(vals)], alpha=0.88, edgecolor='white')

    ymin = max(0, float(np.nanmin(vals)) - 0.08)
    ymax = min(1.0, float(np.nanmax(vals)) + 0.08)
    ax.set_ylim(ymin, ymax)

    ax.set_title(metric, fontsize=10)
    ax.set_xticks(range(len(model_labels)))
    ax.set_xticklabels([m.split('\n')[0] for m in model_labels],
                       fontsize=8, rotation=30, ha='right')

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

fig.suptitle(
    'Model comparison — Review result prediction\n'
    'Red/Orange = text only  |  Green/Blue = tabular only  |  Purple/Teal = fused',
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
kappas = results_df["Cohen's κ"].values.astype(float)
bars = ax.bar(range(len(kappas)), kappas, color=palette[:len(kappas)], alpha=0.88, edgecolor='white')
ax.axhline(0, color='grey', linestyle='--', alpha=0.5, label='Baseline (κ=0, random model)')
ax.axhline(0.2, color='orange', linestyle=':', alpha=0.5, label='κ=0.2 (slight agreement)')
ax.set_ylim(-0.1, 1.05)
ax.set_title("Cohen's κ — Balanced agreement metric (higher = better)\n"
             'κ=0: random model; κ=1: perfect model')
ax.set_xticks(range(len(model_labels)))
ax.set_xticklabels([m.replace('\n', ' ') for m in model_labels],
                   fontsize=8, rotation=20, ha='right')
ax.set_ylabel("Cohen's κ")
ax.legend(fontsize=8)
for bar, v in zip(bars, kappas):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01, f'{v:.3f}',
            ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
radar_metrics = ['Accuracy', 'Macro F1', 'Weighted F1',
                 'Macro Precision', 'Macro Recall']
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + \
         [np.linspace(0, 2 * np.pi, N, endpoint=False)[0]]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for (name, row), color in zip(results_df[radar_metrics].iterrows(), palette):
    values = row.values.tolist() + [row.values[0]]
    ax.plot(angles, values, 'o-', linewidth=1.8, color=color,
            label=name.replace('\n', ' '))
    ax.fill(angles, values, alpha=0.07, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('Model profiles — Radar chart\nOuter = better.',
             fontsize=11, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

---
## 10. Per-Class Analysis

An aggregated metric (Macro F1) can hide serious problems with individual classes.  
For example, a model may have F1=0.5 on average but 0.0 for one of the classes.

Here we analyze the **best model** (by Macro F1) at the level of each result class.

In [ ]:
best_name  = results_df['Macro F1'].idxmax()
worst_name = results_df['Macro F1'].idxmin()
print(f'Best model  : {best_name.replace(chr(10), " ")}')
print(f'Worst model : {worst_name.replace(chr(10), " ")}')

report = classification_report(
    y_test, predictions[best_name],
    target_names=class_names, output_dict=True, digits=3
)
per_class = (
    pd.DataFrame(report).T
    .loc[class_names, ['precision', 'recall', 'f1-score', 'support']]
    .astype(float)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(per_class[['precision', 'recall', 'f1-score']],
            annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, ax=axes[0], linewidths=0.4)
axes[0].set_title(f'Per-class metrics — {best_name.replace(chr(10), " ")}')

colors_bar = ['#27ae60','#2980b9','#f39c12','#e67e22','#e74c3c'][:len(class_names)]
bars = axes[1].barh(
    per_class.index, per_class['f1-score'],
    color=colors_bar, alpha=0.85
)
axes[1].set_xlim(0, 1.35)
axes[1].set_title('F1 by result class (with support)')
for bar, label in zip(bars, per_class.index):
    sup = int(per_class.loc[label, 'support'])
    axes[1].text(bar.get_width() + 0.01,
                 bar.get_y() + bar.get_height() / 2,
                 f'{bar.get_width():.3f}  (n={sup})',
                 va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
### 10.1. Confusion Matrices

The normalized confusion matrix shows **where exactly mispredictions go**. For a perfect model all mass is on the main diagonal.

In our task two characteristic patterns are expected:
- **Tabular models**: errors concentrated near the diagonal
- **Text models**: errors distributed evenly across the matrix

In [ ]:
n_models = len(predictions)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(22, 7 * nrows))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else list(axes)

for ax, (name, y_pred) in zip(axes_flat, predictions.items()):
    cm = confusion_matrix(y_test, y_pred,
                          labels=np.arange(len(classes)), normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.3, vmin=0, vmax=1,
                cbar_kws={'shrink': 0.7})
    ax.set_title(name.replace('\n', ' '), fontsize=9)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)

for ax in axes_flat[n_models:]:
    ax.set_visible(False)

plt.suptitle('Normalized confusion matrices — Review result\n'
             'Perfect model: 1.0 on the main diagonal. Rows = true class.', fontsize=13)
plt.tight_layout()
plt.show()
print('\nText models: errors may be evenly distributed across classes.')
print('Tabular models: better concentration on the diagonal expected.')

---
## 11. Feature Importance

For the `Logistic Regression (Fused)` model we analyze importance by absolute coefficients.

We group features by category:
- **Organizational** : `org_volume`, `org_kind_count`, `org_freq` (Frequency Encoding)
- **Temporal** : `hour`, `dayofweek`, `is_weekend`, `is_night`, sine-cosine encoding
- **Address/Content** : `postcode_known`, `street_len`, `has_building`, `content_len`
- **Categorical (OHE)** : one-hot vectors for `type`, `kind`, `postcode`
- **Text (TF-IDF)** : character n-grams

**Expectation**: thanks to Frequency Encoding and `min_df=5`, the share of categorical and text features should decrease, while organizational features gain relatively more influence.

In [ ]:
# --- model ---
lr_model = models['Logistic Regression (Fused: Text + Tabular)']['model']

# --- coefficients ---
coefs = lr_model.coef_
if coefs.ndim > 1:
    importance = np.mean(np.abs(coefs), axis=0)   # safe for multiclass
else:
    importance = np.abs(coefs)

importances = pd.Series(importance)

# --- feature names (text + tabular) ---
tfidf_names = tfidf.get_feature_names_out()

cat_names = [
    f'{c}={v}'
    for c, vals in zip(CAT_COLS, ohe.categories_)
    for v in vals
]

all_feat_names = list(tfidf_names) + NUM_COLS + cat_names

assert len(all_feat_names) == len(importances), \
    f'Mismatch: {len(all_feat_names)} names vs {len(importances)} importances'

# --- dataframe ---
feat_df = pd.DataFrame({
    'feature': all_feat_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# --- groups ---
temporal_feats = ['hour','dayofweek','is_weekend','is_night','is_business_hours',
                  'hour_sin','hour_cos','dow_sin','dow_cos']
org_feats = ['org_volume', 'org_kind_count', 'org_freq']
address_feats = ['postcode_known', 'street_len', 'has_building', 'content_len']

feat_df['group'] = 'Text (TF-IDF)'

for col in NUM_COLS:
    if col in org_feats:
        feat_df.loc[feat_df['feature'] == col, 'group'] = 'Organizational'
    elif col in temporal_feats:
        feat_df.loc[feat_df['feature'] == col, 'group'] = 'Temporal'
    elif col in address_feats:
        feat_df.loc[feat_df['feature'] == col, 'group'] = 'Address/Content'

feat_df.loc[feat_df['feature'].isin(cat_names), 'group'] = 'Categorical (OHE)'

# --- group importance ---
group_imp = feat_df.groupby('group')['importance'].sum().sort_values(ascending=False)

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors_pie = ['#e74c3c', '#27ae60', '#2980b9', '#f39c12', '#8e44ad']
axes[0].pie(group_imp.values, labels=group_imp.index,
            autopct='%1.1f%%', startangle=90,
            colors=colors_pie[:len(group_imp)],
            textprops={'fontsize': 10})
axes[0].set_title('Feature importance by group (LogReg, text + tabular)')

# --- top features ---
top20 = feat_df.head(20)

color_map = {
    'Organizational': '#e74c3c',
    'Temporal': '#27ae60',
    'Address/Content': '#2980b9',
    'Categorical (OHE)': '#f39c12',
    'Text (TF-IDF)': '#8e44ad'
}

colors_top = [color_map.get(g, 'grey') for g in top20['group']]

axes[1].barh(
    top20['feature'].str[:35][::-1],
    top20['importance'][::-1],
    color=colors_top[::-1],
    alpha=0.85
)

axes[1].set_title('Top-20 most important features')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

# --- output ---
print('\nFeature group importance:')
print(group_imp.round(4))

---
## 12. Error Analysis

This section explores **error patterns** at two levels:

### 12.1. Error Heatmap
For each (class × model) pair, `1 - F1` is shown — the error rate. Darker color = more errors.

### 12.2. Hardest Examples
We find records where **all models were wrong** (`n_correct == 0`). If there are many such records — these examples are genuinely ambiguous even with all signals.

In [ ]:
error_rates = {}
for name, y_pred in predictions.items():
    short = name.replace('\n', ' ')
    report_d = classification_report(
        y_test, y_pred, target_names=class_names,
        output_dict=True, zero_division=0
    )
    error_rates[short] = {
        cls: 1 - report_d[cls]['f1-score'] for cls in class_names
    }

err_df = pd.DataFrame(error_rates)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(err_df, annot=True, fmt='.2f', cmap='Reds',
            vmin=0, vmax=1, ax=ax, linewidths=0.3)
ax.set_title('Error rate (1 - F1) by class × model\n'
             'Darker = more errors. Text models: uniformly dark bands.',
             fontsize=11)
ax.set_xlabel('Model')
ax.set_ylabel('Result class')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
pred_df = pd.DataFrame(
    {name.replace('\n', ' '): y_pred
     for name, y_pred in predictions.items()}
)
pred_df['true_label']  = y_test
pred_df['content']      = test_df['content'].values
pred_df['category']  = test_df['kind'].values
pred_df['hour']     = test_df['hour'].values
pred_df['organization']= test_df['org_id_str'].values

model_cols = [c for c in pred_df.columns
              if c not in ['true_label','content','category','hour','organization']]
pred_df['n_correct']      = (pred_df[model_cols].eq(pred_df['true_label'], axis=0)).sum(axis=1)
pred_df['n_unique_preds'] = pred_df[model_cols].nunique(axis=1)

hardest = pred_df[pred_df['n_correct'] == 0].reset_index(drop=True)
print(f'All models wrong: {len(hardest)} / {len(pred_df)} '
      f'({len(hardest)/len(pred_df)*100:.1f}%)')
print(f'4+ different predictions: '
      f"{len(pred_df[pred_df['n_unique_preds'] >= 4])}")
print('\nHardest record examples:')
# Using print instead of display for nbconvert compatibility
print(hardest[['true_label','category','hour','organization'] + model_cols].head(10).to_string())

---
## 13. Gap Analysis — Where Do Models Diverge?

This section answers: **where exactly does the best model outperform the worst?**

We compute `advantage = best_correct - worst_correct` for each test record and aggregate across three slices:

1. **By hour of day**: are there specific hours where the best model's advantage is particularly visible?
2. **By organization**: top-15 organizations where the gap is largest.
3. **Weekdays vs weekends**: compares accuracy of both models separately.

Green = positive advantage (best model is right), red = negative.

In [ ]:
best_short  = best_name.replace('\n', ' ')
worst_short = worst_name.replace('\n', ' ')

test_df['best_correct']  = (predictions[best_name]  == y_test).astype(int)
test_df['worst_correct'] = (predictions[worst_name] == y_test).astype(int)
test_df['advantage']     = test_df['best_correct'] - test_df['worst_correct']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By hour
hour_adv = test_df.groupby('hour')['advantage'].mean()
axes[0].bar(hour_adv.index, hour_adv.values,
            color=['#27ae60' if v > 0 else '#e74c3c' for v in hour_adv.values],
            alpha=0.85, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Best vs worst: advantage by hour of day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Accuracy gap (best - worst)')

# By organization
org_adv = (test_df.groupby('org_id_str')['advantage'].mean()
           .sort_values(ascending=False).head(15))
axes[1].barh(org_adv.index, org_adv.values,
             color=['#27ae60' if v > 0 else '#e74c3c' for v in org_adv.values],
             alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top-15 orgs where best model dominates')
axes[1].set_xlabel('Accuracy advantage')
axes[1].tick_params(axis='y', labelsize=7)

# Weekdays vs weekends
we_adv = test_df.groupby('is_weekend')[['best_correct', 'worst_correct']].mean()
x = np.arange(2)
w = 0.35
axes[2].bar(x - w/2, we_adv['best_correct'], w,
            label=best_short[:25], color='#27ae60', alpha=0.85)
axes[2].bar(x + w/2, we_adv['worst_correct'], w,
            label=worst_short[:25], color='#e74c3c', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Weekdays', 'Weekends'])
axes[2].set_title('Accuracy: best vs worst on weekdays/weekends')
axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=8)

plt.suptitle(f'Where does {best_short} outperform {worst_short}?', fontsize=12)
plt.tight_layout()
plt.show()

---
## 14. Final Results and Conclusions

In [ ]:
print('=' * 72)
print('FINAL RESULTS — Review result prediction (test: March 2026)')
print('=' * 72)

higher_is_better = list(results_df.columns)
display(results_df.style
        .highlight_max(subset=higher_is_better, axis=0, color='#c8f7c5')
        .highlight_min(subset=higher_is_better, axis=0, color='#f7c5c5')
        .format('{:.4f}'))

gap_f1    = results_df['Macro F1'].max() - results_df['Macro F1'].min()
gap_acc   = results_df['Accuracy'].max() - results_df['Accuracy'].min()
gap_kappa = results_df["Cohen's κ"].max() - results_df["Cohen's κ"].min()

print(f'\n  Best model      : {best_name.replace(chr(10), " ")}')
print(f'  Worst model     : {worst_name.replace(chr(10), " ")}')
print(f'  Macro F1 gap    : {gap_f1:.3f}')
print(f'  Accuracy gap    : {gap_acc:.3f}')
print(f"  Cohen's κ gap   : {gap_kappa:.3f}")
print()
print('Key conclusions (fixed version):')
print('   NaN in result = unreviewed appeals, excluded from analysis')
print('   weekofyear excluded — monotonic feature correlates with time split')
print('   Org aggregates do not use target variable — no leakage')
print('   NB alpha=1.0 — standard Laplace smoothing (fixed from 0.001)')
print('   HistGB max_iter=200, max_depth=4 — real GB power (fixed from 10, 1)')
print('   X_train_fused_dense is now used in HistGB (Dense fused SVD)')
print('   Frequency Encoding for org_id_str instead of OHE — less overfitting')
print('   TF-IDF min_df=5 — noise from rare tokens removed')